# Website topic from HTML structure, language-agnostically

Two models, one pipeline.  The first predicts one of 11 topics from the page's HTML
**without looking at words at all** -- tag tree, attribute usage, link syntax,
metadata/schema, platform fingerprints, class/id naming and a shape profile of the text.
The second adds a **multilingual sentence embedding** of the visible text, which supplies
the content signal while still learning no per-language vocabulary.  Both are measured
against the same three splits, so the cost and the benefit of each are readable.

Everything comes from `structural/`; to rebuild from scratch (run from
`model_training/structural`, details in `README.md`):

```bash
python -m structural.train prep  --data ../27_08_01_en_cz.jsonl --workers 14
python -m structural.embed       --data ../27_08_01_en_cz.jsonl                # ~6 min, cached
python -m structural.train train --data ../27_08_01_en_cz.jsonl --ablation      # feature ladder
python -m structural.train train --data ../27_08_01_en_cz.jsonl --sweep \
    --min-df 4,10,25 --C 1,4,16 --save models/structural_v1.joblib              # structure only
python -m structural.train train --data ../27_08_01_en_cz.jsonl --embed \
    --only struct+embed --min-df 10 --embed-scale 16 \
    --save models/structural_v2_embed.joblib                                    # + text
python -m structural.train explain --model models/structural_v1.joblib --labels Shopping,News
```

Evaluation uses three splits: held-out **websites**, and trained-on-Czech/tested-on-English
plus the reverse (the language-independence test).

In [1]:
import json, sys, gzip, base64
from pathlib import Path
import numpy as np

HERE = Path.cwd() if Path.cwd().name == "structural" else Path("structural")
sys.path.insert(0, str(HERE.parent))
DATA = HERE.parent / "27_08_01_en_cz.jsonl"

from structural import data as D, features as F
from structural.pipeline import StructuralWebsiteClassifier
from structural.train import domain_splits, lang_splits, label_universe, run_config

ds = D.load(DATA, groups=F.ALL_GROUPS, workers=14)
labels, counts = label_universe(ds, min_count=10)
print(len(ds), "pages", ds.stats.shape)
print("languages:", {l: ds.langs.count(l) for l in sorted(set(ds.langs))})
print("domains  :", len(set(ds.domains)))
print("labels   :", {l: counts[l] for l in labels})

[data] loading cached features 27_08_01_en_cz_1e0f1579.joblib


8291 pages (8291, 129)
languages: {'cs': 5476, 'en': 2815}
domains  : 6259
labels   : {'Adult': 127, 'Automotive': 304, 'Computers': 1556, 'Entertainment': 1211, 'Finance': 883, 'Food': 513, 'Health': 813, 'News': 1138, 'Shopping': 2039, 'Sports': 556, 'Travel': 682}


In [2]:
# Which feature families buy what.  `core` = tags/attrs/link-syntax/metadata/platforms,
# i.e. no naming at all; `text_only` is the language-dependent upper bound.
# Hyper-parameters are the ones the sweep picked (reports/sweep.md): min_df=10, C=4,
# raw scores -- per-document z-scoring turned out to hurt once empty predictions
# were fixed by force_one decoding.
runs = {}
for name, groups in [
    ("core", ("tag", "attr", "href", "meta", "plat")),
    ("core+css", ("tag", "attr", "href", "meta", "plat", "css")),
    ("all_struct", F.STRUCT_GROUPS),
    ("text_only", ("words",)),
]:
    r = run_config(ds, name, {"groups": groups, "min_df": 10, "C": 4.0,
                              "score_norm": "none"}, labels, verbose=False)
    runs[name] = r
    print(name, "done", flush=True)

print(f"\n{'config':<12}{'cols':>8}   " + "".join(f"{s:>24}" for s in ("domain", "cs2en", "en2cs")))
for name, r in runs.items():
    cells = ""
    for sp in ("domain", "cs2en", "en2cs"):
        m = r["splits"][sp]
        cells += f"  micro {m['micro_f1']:.3f} auc {m['labeled_mean_auc']:.3f}"
    print(f"{name:<12}{r['X']:>8}{cells}")

core done


core+css done


all_struct done


text_only done



config          cols                     domain                   cs2en                   en2cs
core            7033  micro 0.425 auc 0.765  micro 0.263 auc 0.625  micro 0.234 auc 0.654
core+css       27842  micro 0.467 auc 0.792  micro 0.307 auc 0.668  micro 0.234 auc 0.673
all_struct     50019  micro 0.495 auc 0.807  micro 0.323 auc 0.682  micro 0.281 auc 0.680
text_only      35452  micro 0.757 auc 0.956  micro 0.506 auc 0.875  micro 0.591 auc 0.896


In [3]:
# Per-topic behaviour of the structural model on held-out websites.
import pandas as pd
best = runs["all_struct"]["splits"]["domain"]["per_class"]
pd.DataFrame(best).T.sort_values("n", ascending=False).round(3)

,p,r,f1,n
Shopping,0.724,0.646,0.683,288.0
Entertainment,0.515,0.630,0.567,216.0
News,0.554,0.615,0.583,208.0
Computers,0.467,0.629,0.536,194.0
Health,0.333,0.237,0.277,139.0
Finance,0.270,0.387,0.319,111.0
Sports,0.356,0.564,0.437,110.0
Food,0.414,0.238,0.302,101.0
Travel,0.300,0.219,0.253,96.0
Automotive,0.329,0.418,0.368,55.0


In [4]:
# How much of the cross-lingual loss is the representation and how much is only the
# decision thresholds?  A model moved from English to Czech pages keeps its *ranking*
# (AUC), but per-label cut-offs tuned on English validation rows stop fitting because the
# whole score level shifts.  Re-tuning the cut-offs on target-language rows -- weights and
# features untouched -- separates the two effects.  Rows are split in half so the re-tuned
# thresholds are still evaluated on pages they never saw.
from sklearn.metrics import roc_auc_score
from structural.model import binarize, tune_thresholds, report

tr, va, te = lang_splits(ds)["en2cs"]
clf_e2c = StructuralWebsiteClassifier.fit(ds, tr, va, labels, groups=F.STRUCT_GROUPS,
                                       min_df=10, C=4.0, verbose=False)
S = clf_e2c.score_matrix(ds, te)
Y = binarize([ds.labels[i] for i in te], labels)
pick = np.random.default_rng(0).permutation(len(te))
A, B = pick[: len(te) // 2], pick[len(te) // 2:]        # A = re-tuning rows, B = scored

def show(tag, p):
    m = report(Y[B], p, labels)
    print(f"  {tag:<36} micro {m['micro_f1']:.3f}  macro {m['macro_f1']:.3f}  "
          f"exact {m['exact_match']:.3f}")

for k in (1, 2):                                          # threshold-free decoding
    p = np.zeros_like(Y[B])
    p[np.arange(len(B))[:, None], np.argsort(-S[B], 1)[:, :k]] = 1
    show(f"top-{k} (no thresholds)", p)

def decoded(thr):
    p = (S[B] >= np.asarray(thr)).astype(np.int8)
    empty = p.sum(1) == 0
    p[empty, S[B][empty].argmax(1)] = 1                   # same force_one rule as predict()
    return p

show("thresholds tuned on English rows", decoded(clf_e2c.thresholds))
show("thresholds re-tuned on Czech rows", decoded(tune_thresholds(S[A], Y[A])))
print(f"\n  mean AUC of the scored half (threshold-free): "
      f"{roc_auc_score(Y[B], S[B], average=None).mean():.3f}")

  top-1 (no thresholds)                micro 0.356  macro 0.274  exact 0.317
  top-2 (no thresholds)                micro 0.348  macro 0.298  exact 0.016
  thresholds tuned on English rows     micro 0.276  macro 0.284  exact 0.067


  thresholds re-tuned on Czech rows    micro 0.334  macro 0.319  exact 0.115

  mean AUC of the scored half (threshold-free): 0.674


## Content, without a vocabulary

The structural features name page *function*, not subject: `Shopping` survives a change of
language because a product grid plus a price plus a cart form is a layout, while `Food`
collapses to 0.031 because cuisine is content.  The fix is a **multilingual sentence
embedding** of the visible text -- `paraphrase-multilingual-MiniLM-L12-v2`, 384 dims, one
shared space in which Czech and English translations land on top of each other (0.84-0.95
cosine on this corpus, 0.11 for an unrelated pair).  Nothing in the pipeline holds a
per-language vocabulary: the vectors go in as a dense block after an SVD, so the
classifier still never sees a word.

In [5]:
# Same three splits, same hyper-parameters as the rows above.  `all_struct` and
# `text_only` are the fits from the ablation cell rather than a second run -- their
# feature rows do not involve the text block at all.
from structural.embed import DEFAULT_MODEL

ds2 = D.load(DATA, groups=F.ALL_GROUPS, workers=14, embed_model=DEFAULT_MODEL)
print("groups:", ds2.groups, "| embeds:", tuple(ds2.embeds.shape))

cmp_runs = {k: runs[k] for k in ("all_struct", "text_only")}
for name, groups in [
    ("embed_only", ("embed",)),                           # content only
    ("struct+embed", F.STRUCT_GROUPS + ("embed",)),       # shipped as v2
    ("struct+text+embed", F.STRUCT_GROUPS + ("words", "embed")),
]:
    cmp_runs[name] = run_config(ds2, name, {"groups": groups, "min_df": 10, "C": 4.0,
                                            "embed_scale": 16.0}, labels, verbose=False)
    print(name, "done", flush=True)

# `--embed-scale 16` is not a detail: concatenated at 1.0 the 128 dense dims are ~0.4% of
# each row's L2 norm next to 50k tf-idf columns, and cs2en scores 0.369 instead of 0.678.
# one row per model, one column per split metric  (a dict of dicts comes out with the
# model names as *columns*, hence the transpose)
pd.DataFrame({n: {f"{sp} micro": m["micro_f1"] for sp, m in r["splits"].items()}
                 | {f"{sp} auc": m["labeled_mean_auc"] for sp, m in r["splits"].items()}
              for n, r in cmp_runs.items()}).T.round(3).sort_values("cs2en micro",
                                                                    ascending=False)

[data] loading cached features 27_08_01_en_cz_1e0f1579.joblib


[embed] attached (8291, 384) from embed_27_08_01_en_cz_d2587d86.npz
groups: ['tag', 'attr', 'href', 'meta', 'marker', 'plat', 'res', 'css', 'cssraw', 'shape', 'url', 'words', 'embed'] | embeds: (8291, 384)


embed_only done


struct+embed done


struct+text+embed done


,domain micro,cs2en micro,en2cs micro,domain auc,cs2en auc,en2cs auc
struct+text+embed,0.734,0.678,0.674,0.950,0.921,0.914
struct+embed,0.704,0.678,0.654,0.941,0.919,0.910
embed_only,0.681,0.649,0.641,0.927,0.909,0.907
text_only,0.757,0.506,0.591,0.956,0.875,0.896
all_struct,0.495,0.323,0.281,0.807,0.682,0.680


In [6]:
# Where the text block earns its keep: per-class F1 on held-out websites and on the
# Czech->English transfer, structure only vs structure + text.
base, plus = runs["all_struct"], cmp_runs["struct+embed"]


def perclass(r, sp):
    return pd.Series({l: v["f1"] for l, v in r["splits"][sp]["per_class"].items()})


out = pd.DataFrame({"struct": perclass(base, "domain"),
                    "+ text": perclass(plus, "domain"),
                    "struct cs2en": perclass(base, "cs2en"),
                    "+ text cs2en": perclass(plus, "cs2en")})
out["delta (held-out sites)"] = out["+ text"] - out["struct"]
out["delta (new language)"] = out["+ text cs2en"] - out["struct cs2en"]
out.sort_values("delta (held-out sites)", ascending=False).round(3)

,struct,+ text,struct cs2en,+ text cs2en,delta (held-out sites),delta (new language)
Health,0.277,0.704,0.165,0.686,0.427,0.521
Food,0.302,0.714,0.031,0.567,0.412,0.536
Automotive,0.368,0.739,0.058,0.524,0.371,0.466
Travel,0.253,0.547,0.134,0.575,0.294,0.441
Finance,0.319,0.607,0.170,0.584,0.288,0.414
Computers,0.536,0.747,0.406,0.724,0.211,0.318
Sports,0.437,0.634,0.202,0.707,0.197,0.505
Entertainment,0.567,0.725,0.308,0.711,0.158,0.403
Adult,0.786,0.903,0.066,0.891,0.117,0.825
News,0.583,0.679,0.238,0.520,0.096,0.282


## The same pages, scored by both bundles

`v1` reads structure only, so it needs no text model at serving time; `v2` adds the
multilingual vectors and costs a torch model on the serving host.  Both are handed nothing
but the HTML bytes of a page.

In [7]:
v1 = StructuralWebsiteClassifier.load(HERE / "models" / "structural_v1.joblib")
v2 = StructuralWebsiteClassifier.load(HERE / "models" / "structural_v2_embed.joblib")
print("labels      :", v1.labels)
print("v1 groups   :", v1.groups)
print("v1 vocab    :", len(v1.vectorizer.vocab), "tokens +",
      len(v1.vectorizer.num_names), "statistics")
print("v2 groups   :", v2.groups)
print("v2 encoder  :", v2.vectorizer.embed_model.split("/")[-1])
print("v2 columns  :", v2.vectorizer.n_cols_, "of which", v2.vectorizer.emb_dim_,
      "are text

SyntaxError: unterminated string literal (detected at line 10) (1057994628.py, line 10)

In [ ]:
# Two Czech and two English pages straight out of the dataset; `truth` is the label the
# corpus carries, the two answers below come from the two bundles.

want, seen = {"cs": 2, "en": 2}, []
with open(DATA, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        lang = D.detect_lang(D.decompress(rec["compressed_html"]), rec["url"])
        if want.get(lang, 0) > 0:
            want[lang] -= 1
            seen.append((lang, rec))
        if sum(want.values()) == 0:
            break

def fmt(p):
    return ", ".join(f"{lab} {s:.2f}" for lab, s in p)

for lang, rec in seen:
    raw = D.decompress(rec["compressed_html"])
    print(f"\n{lang}  {rec['url']}\n  truth       : {rec['categories']}"
          f"\n  structure   : {fmt(v1.predict(raw, url=rec['url'])[:2])}"
          f"\n  + text      : {fmt(v2.predict(raw, url=rec['url'])[:2])}")

## Reading the numbers

- **Structure alone is worth something, but it is not free.** Word-free, on held-out
  websites: micro F1 0.495 / AUC 0.807, against 0.757 / 0.956 for a baseline allowed to
  read the words. A third of the signal really is in the markup.
- **Most of that third comes from naming, not from the tree.** Tags alone (0.332) do no
  better than the 129 raw page statistics (0.344). Attributes make it work (0.413) and
  CSS class names are the single biggest addition (0.473) -- CMS vocabulary (`product`,
  `price`, `cart`, `article`, `date`) is English on Czech, German and Korean sites alike,
  which is why it counts as language-free.
- **Structure names page *function*, not subject.** `Shopping` survives the language
  switch (F1 0.571 Czech→English) because a product grid plus a price plus a cart form
  *is* a layout; `Food` collapses to 0.031 because cuisine is content.
- **Content can be added without adding a language, and it is the better model.** A
  multilingual sentence embedding lifts held-out websites 0.495 → 0.704 and unseen
  languages 0.323 → 0.678 (`cs2en`), and it is the content classes that move: `Food`
  0.302 → 0.714, `Health` 0.277 → 0.704, `Automotive` 0.368 → 0.739. Unlike word TF-IDF
  it loses only 0.03 micro F1 to a change of language, because the encoder puts
  translations on top of each other -- and unlike word TF-IDF it will still work on a
  language it never saw.
- **Watch `--embed-scale` when mixing dense blocks with tf-idf.** At the natural scale of
  1.0 the text block is 0.4% of each row's norm and the combined model scores 0.369 on
  `cs2en` -- far worse than the same vectors used on their own (0.649). At scale 16 it
  reaches 0.678. Any dense-plus-sparse model needs this knob set deliberately; the
  default of 1.0 is not it.
- **Part of the cross-lingual loss is only calibration.** With the weights frozen,
  re-tuning thresholds on target-language rows recovers 0.06 micro F1 and plain top-1
  decoding beats the source-language thresholds outright (0.356 vs 0.276). Deployed on a
  new language: use top-k, do not ship the tuned thresholds.
- **`cs2en`/`en2cs` are a worst case, not a clean language ablation.** The two halves
  differ in vintage and CMS generation as well as language -- the English half is a 2004
  archive, and the model exploits that (see the `font` and `td[bgcolor]` weights under
  `News`), which flatters `domain` and punishes the cross-lingual columns.